In [18]:
import pandas as pd
import time
from google import genai #Check API usage & rate limits at: https://aistudio.google.com/projects
from google.genai import types
import CJDH_local_settings

In [19]:
# client = genai.Client(api_key=CJDH_local_settings.local_settings['GenAI_Settings']['google_genai_API_KEY'])
# MODEL_ID = "gemini-2.5-flash-lite"

# df = pd.read_csv('2526players_mids.csv')

# def classify_batch_minimal(player_names):
#     names_input = ",".join(player_names)

#     prompt = f"Classify each midfielder as Defensive (D), Attacking (A) or Unclear (U). Output only: D,A,U...\n{names_input}"

#     for attempt in range(3):
#         response = client.models.generate_content(
#             model=MODEL_ID,
#             contents=prompt,
#             config=types.GenerateContentConfig(
#                 temperature=0.0,
#                 max_output_tokens=2001
#             )
#         )
        
#         labels = response.text.strip().replace(" ", "").split(",")
        
#         if len(labels) == len(player_names) and all(l in ['D','A','U'] for l in labels):
#             return labels
        
#         if attempt < 2:
#             time.sleep(1)
    
#     print(f"Failed batch of {len(player_names)}, marking as U")
#     return ['U'] * len(player_names)

# # Execution
# batch_size = 1000
# all_labels = []
# player_list = df['player_name_id'].tolist()

# for i in range(0, len(player_list), batch_size):
#     batch = player_list[i:i+batch_size]
#     print(f"Batch {i//batch_size + 1}: {len(batch)} players")
#     labels = classify_batch_minimal(batch)
#     all_labels.extend(labels)
#     time.sleep(4)

# df['position_code'] = all_labels

In [20]:
import json

client = genai.Client(api_key=CJDH_local_settings.local_settings['GenAI_Settings']['google_genai_API_KEY'])
MODEL_ID = "gemini-2.5-flash-lite"

df = pd.read_csv('2526players_mids.csv')

def classify_batch_minimal(player_names):
    # Numbering the list helps the model keep track of the count
    formatted_list = "\n".join([f"{i+1}. {name}" for i, name in enumerate(player_names)])
    
    prompt = f"""Classify these {len(player_names)} midfielders as:
                    D (Defensive)
                    A (Attacking)
                    U (Unclear)

                    Output a JSON object with a key "labels" containing a list of codes in order.
                    Example: {{"labels": ["D", "A", "U"]}}

                    Players:
                    {formatted_list}"""

    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.0,
                    response_mime_type="application/json", # Forces JSON format
                )
            )
            
            # Parse the JSON response
            data = json.loads(response.text)
            labels = data.get("labels", [])

            if len(labels) == len(player_names):
                return labels
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
        
        time.sleep(1)
    
    print(f"Failed batch of {len(player_names)}, marking as U")
    return ['U'] * len(player_names)


import time

# Configuration
batch_size = 50  # Recommended for better accuracy/reliability
request_count = 0
all_labels = []
player_list = df['player_name_id'].tolist()

for i in range(0, len(player_list), batch_size):
    batch = player_list[i : i + batch_size]
    batch_number = (i // batch_size) + 1
    
    print(f"Processing Batch {batch_number} ({len(batch)} players)...")
    
    # Execute classification
    labels = classify_batch_minimal(batch)
    all_labels.extend(labels)
    
    request_count += 1
    
    # Rate Limit Handling: Every 10 requests, wait 61 seconds
    if request_count % 10 == 0 and i + batch_size < len(player_list):
        print(f"--- Reached 10 requests. Pausing for 61 seconds to reset rate limits... ---")
        time.sleep(61)
    else:
        # Standard small delay between individual requests to avoid burst errors
        time.sleep(2)

df['position_code'] = all_labels

Processing Batch 1 (50 players)...
Processing Batch 2 (50 players)...
Processing Batch 3 (50 players)...
Processing Batch 4 (50 players)...
Processing Batch 5 (50 players)...
Processing Batch 6 (50 players)...
Processing Batch 7 (50 players)...
Processing Batch 8 (11 players)...


In [21]:
df['position_code'].value_counts()

position_code
A    202
U     85
D     74
Name: count, dtype: int64

In [25]:
df[df['position_code'] == 'A'].head()

,player_name_id,team_name,position_code
0,Aaron Ramsey,Burnley,A
1,Abdallah Sima,Brighton,A
2,Abdoullah Ba,Sunderland,A
4,Adama Traoré Diarra,Fulham,A
5,Adrian Mazilu,Brighton,A


In [26]:
df[df['position_code'] == 'D'].head()

,player_name_id,team_name,position_code
7,Albert Sambi Lokonga,Arsenal,D
15,Amadou Onana,Aston Villa,D
21,André Trindade da Costa Neto,Wolves,D
26,Anton Stach,Leeds,D
38,Boubacar Kamara,Aston Villa,D


In [27]:
df[df['position_code'] == 'U'].head()

,player_name_id,team_name,position_code
3,Adam Wharton,Crystal Palace,U
6,Alan Browne,Sunderland,U
10,Alex Scott,Bournemouth,U
11,Alex Tóth,Bournemouth,U
13,Alysson Edward Franco da Rocha dos Santos,Aston Villa,U
